In [0]:
CREATE VOLUME IF NOT EXISTS workspace.default.lantern_walks;

In [0]:
%python
from pathlib import Path

WALK_ROOT = Path("/Volumes/workspace/default/lantern_walks")

for name in ["incoming", "rgb", "depth", "masks", "exports"]:
    (WALK_ROOT / name).mkdir(parents=True, exist_ok=True)

print("Lantern walk storage ready")

for folder in WALK_ROOT.iterdir():
    print(folder)

In [0]:

%python
from pathlib import Path
from pyspark.sql import functions as F, Window

ROOT = Path("/Volumes/workspace/default/lantern_walks")
INCOMING = ROOT / "incoming"
DB = "workspace.default"

BRONZE = f"{DB}.lantern_bronze"
SILVER = f"{DB}.lantern_silver"
GOLD = f"{DB}.lantern_gold_hazards"
REVIEW = f"{DB}.lantern_review_candidates"
TRACKS = f"{DB}.lantern_pedestrian_tracks"
SESSIONS = f"{DB}.lantern_sessions"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE} (
    record_hash STRING,
    raw_json STRING,
    source_file STRING,
    ingested_at TIMESTAMP
) USING DELTA
""")

incoming_files = sorted(
    str(p)
    for p in INCOMING.rglob("*.jsonl")
    if p.is_file()
    and not p.name.upper().startswith("SIM_")
)

if incoming_files:
    new = (
        spark.read.text(incoming_files)
        .select(
            F.col("value").alias("raw_json"),
            F.col("_metadata.file_path").alias("source_file")
        )
        .where(F.length(F.trim("raw_json")) > 0)
        .withColumn("record_hash", F.sha2("raw_json", 256))
        .withColumn("ingested_at", F.current_timestamp())
        .dropDuplicates(["record_hash"])
    )

    new.createOrReplaceTempView("lantern_new_records")

    spark.sql(f"""
        MERGE INTO {BRONZE} t
        USING lantern_new_records s
        ON t.record_hash = s.record_hash
        WHEN NOT MATCHED THEN INSERT (
            record_hash,
            raw_json,
            source_file,
            ingested_at
        )
        VALUES (
            s.record_hash,
            s.raw_json,
            s.source_file,
            s.ingested_at
        )
    """)

frame_schema = """
source_type STRING,
session_id STRING,
frame_id BIGINT,
captured_at_ms BIGINT,
rgb_path STRING,
depth_path STRING,
rgb_ts_ms BIGINT,
depth_ts_ms BIGINT,
depth_valid_fraction DOUBLE,
camera_pose STRUCT<
    tx_m:DOUBLE,
    ty_m:DOUBLE,
    tz_m:DOUBLE,
    qx:DOUBLE,
    qy:DOUBLE,
    qz:DOUBLE,
    qw:DOUBLE
>,
glass_model_version STRING,
stair_model_version STRING,
people_model_version STRING,
glass_probability DOUBLE,
stair_riser_probability DOUBLE,
stair_tread_probability DOUBLE,
glass_mask_path STRING,
stair_mask_path STRING,
people ARRAY<STRUCT<
    track_id:STRING,
    confidence:DOUBLE,
    bbox_xywh:ARRAY<DOUBLE>,
    coordinate_frame:STRING,
    x_m:DOUBLE,
    z_m:DOUBLE,
    vx_mps:DOUBLE,
    vz_mps:DOUBLE
>>
"""

w = Window.partitionBy(
    "session_id", "frame_id"
).orderBy(
    F.desc("ingested_at"),
    F.desc("record_hash")
)

silver = (
    spark.table(BRONZE)
    .withColumn("frame", F.from_json("raw_json", frame_schema))
    .select(
        "record_hash",
        "source_file",
        "ingested_at",
        "frame.*"
    )
    .where(
        (F.col("source_type") == "iphone_recording")
        & F.col("session_id").isNotNull()
        & F.col("frame_id").isNotNull()
        & F.col("captured_at_ms").isNotNull()
    )
    .withColumn("_rank", F.row_number().over(w))
    .where(F.col("_rank") == 1)
    .drop("_rank")
    .withColumn(
        "rgb_depth_gap_ms",
        F.abs(F.col("rgb_ts_ms") - F.col("depth_ts_ms"))
    )
    .withColumn(
        "timestamps_close",
        F.coalesce(
            F.col("rgb_depth_gap_ms") <= 80,
            F.lit(False)
        )
    )
    .withColumn(
        "people_count",
        F.coalesce(F.size("people"), F.lit(0))
    )
)

asset_paths = [
    r["path"]
    for r in (
        silver.select(F.col("rgb_path").alias("path"))
        .union(
            silver.select(F.col("depth_path").alias("path"))
        )
    ).where(F.col("path").isNotNull()).distinct().collect()
]

if len(asset_paths) > 50000:
    raise RuntimeError("Too many asset paths in one batch")

flags = spark.createDataFrame(
    [
        (
            p,
            p.startswith(str(ROOT) + "/")
            and Path(p).is_file()
        )
        for p in asset_paths
    ],
    "path STRING, exists BOOLEAN"
)

rgb_flags = flags.select(
    F.col("path").alias("rgb_path"),
    F.col("exists").alias("rgb_exists")
)

depth_flags = flags.select(
    F.col("path").alias("depth_path"),
    F.col("exists").alias("depth_exists")
)

silver = (
    silver
    .join(F.broadcast(rgb_flags), "rgb_path", "left")
    .join(F.broadcast(depth_flags), "depth_path", "left")
    .fillna({
        "rgb_exists": False,
        "depth_exists": False
    })
    .withColumn(
        "rgb_depth_aligned",
        F.col("rgb_exists")
        & F.col("depth_exists")
        & F.col("timestamps_close")
    )
    .withColumn(
        "data_quality",
        F.when(
            F.col("rgb_depth_aligned"),
            "metadata_pair_valid"
        )
        .when(
            ~F.col("rgb_exists") | ~F.col("depth_exists"),
            "missing_file"
        )
        .otherwise("timestamp_mismatch")
    )
)

def save(df, table):
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )

save(silver, SILVER)
silver = spark.table(SILVER)

stair_score = F.greatest(
    "stair_riser_probability",
    "stair_tread_probability"
)

def hazard_rows(kind, condition, score, version):
    return silver.where(condition).select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "rgb_path",
        "depth_path",
        "rgb_depth_aligned",
        "source_file",
        F.lit(kind).alias("hazard_type"),
        score.cast("double").alias("confidence"),
        F.col(version).alias("model_version")
    )

glass_events = hazard_rows(
    "glass",
    F.col("glass_probability") >= 0.65,
    F.col("glass_probability"),
    "glass_model_version"
)

stair_events = hazard_rows(
    "stairs",
    stair_score >= 0.65,
    stair_score,
    "stair_model_version"
)

person_events = hazard_rows(
    "person",
    F.col("people_count") > 0,
    F.lit(None),
    "people_model_version"
)

gold = (
    glass_events
    .unionByName(stair_events)
    .unionByName(person_events)
)

save(gold, GOLD)

reasons = F.array(
    F.when(
        ~F.col("rgb_depth_aligned"),
        F.lit("missing_or_misaligned_sensor_files")
    ),
    F.when(
        F.col("glass_probability").between(0.35, 0.65),
        F.lit("uncertain_glass")
    ),
    F.when(
        stair_score.between(0.35, 0.65),
        F.lit("uncertain_stairs")
    ),
    F.when(
        (F.col("glass_probability") >= 0.65)
        & (F.col("depth_valid_fraction") < 0.5),
        F.lit("glass_low_depth_validity")
    )
)

reviews = (
    silver
    .withColumn("reason", F.explode(reasons))
    .where(F.col("reason").isNotNull())
    .select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "rgb_path",
        "depth_path",
        "reason",
        "glass_probability",
        "depth_valid_fraction",
        "source_file"
    )
)

save(reviews, REVIEW)

track_w = Window.partitionBy(
    "session_id",
    "track_id"
).orderBy(
    "captured_at_ms",
    "frame_id"
)

tracks = (
    silver
    .select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "source_file",
        F.explode_outer("people").alias("person")
    )
    .where(F.col("person.track_id").isNotNull())
    .select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "source_file",
        F.col("person.track_id").alias("track_id"),
        F.col("person.confidence").alias("confidence"),
        F.col("person.bbox_xywh").alias("bbox_xywh"),
        F.col("person.coordinate_frame").alias("coordinate_frame"),
        F.col("person.x_m").alias("x_m"),
        F.col("person.z_m").alias("z_m"),
        F.col("person.vx_mps").alias("supplied_vx"),
        F.col("person.vz_mps").alias("supplied_vz")
    )
    .withColumn(
        "dt",
        (
            F.col("captured_at_ms")
            - F.lag("captured_at_ms").over(track_w)
        ) / 1000
    )
    .withColumn("prev_x", F.lag("x_m").over(track_w))
    .withColumn("prev_z", F.lag("z_m").over(track_w))
    .withColumn(
        "prev_frame",
        F.lag("coordinate_frame").over(track_w)
    )
    .withColumn(
        "world_ok",
        (F.col("coordinate_frame") == "arkit_world")
        & F.col("x_m").isNotNull()
        & F.col("z_m").isNotNull()
    )
    .withColumn(
        "motion_ok",
        F.col("world_ok")
        & (F.col("prev_frame") == "arkit_world")
        & F.col("dt").between(0.001, 2.0)
    )
    .withColumn(
        "vx_mps",
        F.when(
            F.col("world_ok"),
            F.coalesce(
                "supplied_vx",
                F.when(
                    F.col("motion_ok"),
                    (
                        F.col("x_m") - F.col("prev_x")
                    ) / F.col("dt")
                )
            )
        )
    )
    .withColumn(
        "vz_mps",
        F.when(
            F.col("world_ok"),
            F.coalesce(
                "supplied_vz",
                F.when(
                    F.col("motion_ok"),
                    (
                        F.col("z_m") - F.col("prev_z")
                    ) / F.col("dt")
                )
            )
        )
    )
    .withColumn(
        "predicted_x_m_1s",
        F.when(
            F.col("world_ok"),
            F.col("x_m") + F.col("vx_mps")
        )
    )
    .withColumn(
        "predicted_z_m_1s",
        F.when(
            F.col("world_ok"),
            F.col("z_m") + F.col("vz_mps")
        )
    )
    .select(
        "session_id",
        "frame_id",
        "captured_at_ms",
        "track_id",
        "confidence",
        "bbox_xywh",
        "coordinate_frame",
        "x_m",
        "z_m",
        "vx_mps",
        "vz_mps",
        "predicted_x_m_1s",
        "predicted_z_m_1s",
        "source_file"
    )
)

save(tracks, TRACKS)

sessions = (
    silver.groupBy("session_id")
    .agg(
        F.count("*").alias("frame_count"),
        F.sum(
            F.col("rgb_depth_aligned").cast("int")
        ).alias("metadata_aligned_frames"),
        F.avg(
            "depth_valid_fraction"
        ).alias("average_depth_valid_fraction")
    )
    .join(
        spark.table(GOLD)
        .groupBy("session_id")
        .agg(F.count("*").alias("hazard_events")),
        "session_id",
        "left"
    )
    .join(
        spark.table(REVIEW)
        .groupBy("session_id")
        .agg(F.count("*").alias("review_candidates")),
        "session_id",
        "left"
    )
    .fillna({
        "hazard_events": 0,
        "review_candidates": 0
    })
)

save(sessions, SESSIONS)

print("LANTERN LAKEHOUSE READY")

for table in [
    BRONZE,
    SILVER,
    GOLD,
    REVIEW,
    TRACKS,
    SESSIONS
]:
    print(
        table,
        spark.table(table).count(),
        "rows"
    )

if not incoming_files:
    print(
        "Waiting for real iPhone recordings in:",
        INCOMING
    )
else:
    print(
        "Real recording files found:",
        len(incoming_files)
    )
